# ROGII TVT Model — Train, Freeze & Export
**Purpose:** Train a LightGBM model to predict True Vertical Thickness (TVT) along horizontal wellbores,
using GR-typewell correlation features. Export to both pickle (Python runtime) and ONNX (edge/Drillbotics deployment).

**Pipeline stages:**
1. Data loading & structure inspection
2. Typewell interpolation (align typewell GR to lateral MD grid)
3. Feature engineering (rolling GR stats + DTW correlation lag)
4. Group K-Fold CV (split by well_id — never by row)
5. LightGBM training with monotonic constraint on depth
6. Savitzky-Golay post-processing (enforce physical smoothness)
7. Export: pickle pipeline + ONNX
8. Sanity-check: run a single-well inference in <5ms

**Drillbotics note:** The exported ONNX model is the geological state estimator that feeds the
Case 2 geosteering controller via D-WIS/OPC-UA. It takes a live GR window as input and
outputs an estimated TVT position in real time.

## 0. Install & imports

In [ ]:
# Run once in Kaggle — most are pre-installed, dtaidistance is not
!pip install dtaidistance skl2onnx onnxruntime --quiet

In [ ]:
import os, glob, warnings, pickle, time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d

import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

# DTW — fast C implementation
from dtaidistance import dtw

# ONNX export
from skl2onnx import convert_sklearn, update_registered_converter
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/input/rogii-wellbore-geology-prediction')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
N_FOLDS = 5

## 1. Data loading & structure inspection

The competition data has a `train/` folder with one CSV per well (or a combined CSV).
We auto-detect both layouts below.

In [ ]:
def discover_data(data_dir: Path):
    """
    Auto-detect whether data is:
      (a) one combined train.csv  +  typewell.csv / typewell_logs.csv
      (b) train/ folder with per-well CSV files
    Returns: train_df, typewell_df
    """
    print("Files found:")
    for f in sorted(data_dir.rglob('*.csv')):
        print(f"  {f.relative_to(data_dir)}  ({f.stat().st_size//1024} KB)")

    # ── combined layout ────────────────────────────────────────────────────
    train_candidates = list(data_dir.glob('train.csv')) + list(data_dir.glob('train/*.csv'))
    tw_candidates    = (list(data_dir.glob('typewell*.csv')) +
                        list(data_dir.glob('*typewell*.csv')) +
                        list(data_dir.glob('train/typewell*.csv')))

    # Load lateral wells
    if (data_dir / 'train.csv').exists():
        train_df = pd.read_csv(data_dir / 'train.csv')
    elif (data_dir / 'train').is_dir():
        frames = []
        for f in sorted((data_dir / 'train').glob('*.csv')):
            df = pd.read_csv(f)
            # inject well_id from filename if column absent
            if 'well_id' not in df.columns:
                df.insert(0, 'well_id', f.stem)
            frames.append(df)
        train_df = pd.concat(frames, ignore_index=True)
    else:
        raise FileNotFoundError(f"No train data found under {data_dir}")

    # Load typewell
    if tw_candidates:
        typewell_df = pd.read_csv(tw_candidates[0])
    else:
        typewell_df = None
        print("WARNING: no typewell file found — typewell features will be skipped")

    return train_df, typewell_df


train_df, typewell_df = discover_data(DATA_DIR)
print(f"\nTrain shape : {train_df.shape}")
print(f"Train columns: {train_df.columns.tolist()}")
print(train_df.head(3))

if typewell_df is not None:
    print(f"\nTypewell shape : {typewell_df.shape}")
    print(f"Typewell columns: {typewell_df.columns.tolist()}")
    print(typewell_df.head(3))

In [ ]:
# ── Normalise column names to snake_case ──────────────────────────────────
# The competition may use various casings; standardise here.
def normalise_cols(df):
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r'[\s\-\.]+', '_', regex=True)
    )
    return df

train_df    = normalise_cols(train_df)
if typewell_df is not None:
    typewell_df = normalise_cols(typewell_df)

# ── Identify key columns (handle common naming variants) ─────────────────
COL_WELL  = next((c for c in train_df.columns if 'well' in c and 'id' in c), 'well_id')
COL_MD    = next((c for c in train_df.columns if c in ('md', 'measured_depth', 'depth_md', 'md_m')), 'md')
COL_GR    = next((c for c in train_df.columns if c in ('gr', 'gamma_ray', 'gr_api')), 'gr')
COL_TVT   = next((c for c in train_df.columns if 'tvt' in c), 'tvt')

print(f"Mapped columns → well:{COL_WELL}  md:{COL_MD}  gr:{COL_GR}  tvt:{COL_TVT}")
print(f"\nWell IDs ({train_df[COL_WELL].nunique()} wells):")
print(train_df[COL_WELL].value_counts())
print(f"\nTVT range: {train_df[COL_TVT].min():.2f} → {train_df[COL_TVT].max():.2f}")

## 2. Typewell interpolation

The typewell is a vertical reference well sampled at its own depth grid.
We interpolate its GR onto each lateral well's MD grid so we can compute
GR cross-correlation features at every depth step.

In [ ]:
def interpolate_typewell_gr(typewell_df, lateral_md: np.ndarray) -> np.ndarray:
    """
    Linearly interpolate typewell GR onto the lateral's MD array.
    Assumes the typewell depth column maps to lateral MD after TVD correction
    (simplification — a full TVD↔MD transform would use the survey data).
    """
    if typewell_df is None:
        return np.full(len(lateral_md), np.nan)

    tw_depth_col = next(
        (c for c in typewell_df.columns if c in ('depth', 'tvd', 'md', 'depth_m')), None
    )
    tw_gr_col    = next(
        (c for c in typewell_df.columns if c in ('gr', 'gamma_ray', 'gr_api')), None
    )

    if tw_depth_col is None or tw_gr_col is None:
        print(f"  Typewell: couldn't resolve depth/GR cols from {typewell_df.columns.tolist()}")
        return np.full(len(lateral_md), np.nan)

    tw = typewell_df[[tw_depth_col, tw_gr_col]].dropna().sort_values(tw_depth_col)
    f  = interp1d(
        tw[tw_depth_col].values,
        tw[tw_gr_col].values,
        bounds_error=False,
        fill_value=(tw[tw_gr_col].iloc[0], tw[tw_gr_col].iloc[-1])
    )
    return f(lateral_md)


# Quick visual check on one well
sample_well = train_df[COL_WELL].unique()[0]
sample = train_df[train_df[COL_WELL] == sample_well].sort_values(COL_MD)
tw_interp = interpolate_typewell_gr(typewell_df, sample[COL_MD].values)
print(f"Typewell GR interpolated for well '{sample_well}': {tw_interp[:5]}")

## 3. Feature engineering

Three feature groups:

**A — Depth / trajectory features** (positional context)  
**B — Rolling GR statistics** (local signal shape at multiple scales)  
**C — GR–typewell correlation features** (the core geosteering matching signal)

The DTW best-lag feature is the most important single feature — it encodes
"how far in the typewell does the current lateral GR pattern best match?"
which is exactly what TVT is telling you.

In [ ]:
# Rolling windows (in depth samples, assuming ~1 ft spacing)
WINDOWS = [5, 15, 30, 60, 120]

# Correlation lag search range (samples)
CORR_LAGS   = list(range(-60, 61, 5))   # ±60 ft in 5 ft steps
DTW_WINDOW  = 30                         # DTW local window half-width


def rolling_gr_features(gr: np.ndarray, md: np.ndarray, windows=WINDOWS) -> pd.DataFrame:
    """Group A + B: depth context and rolling GR statistics."""
    s   = pd.Series(gr)
    out = {'md': md}

    # Depth features
    out['md_norm']           = (md - md.min()) / (md.max() - md.min() + 1e-9)
    out['dist_from_heel_m']  = md - md.min()

    # GR raw + gradient
    out['gr']       = gr
    out['gr_grad']  = np.gradient(gr, md)
    out['gr_grad2'] = np.gradient(out['gr_grad'], md)

    # Rolling statistics
    for w in windows:
        r = s.rolling(w, center=True, min_periods=1)
        out[f'gr_mean_{w}']   = r.mean().values
        out[f'gr_std_{w}']    = r.std().fillna(0).values
        out[f'gr_min_{w}']    = r.min().values
        out[f'gr_max_{w}']    = r.max().values
        out[f'gr_range_{w}']  = out[f'gr_max_{w}'] - out[f'gr_min_{w}']

    # Percentile rank within well
    out['gr_pct_rank'] = pd.Series(gr).rank(pct=True).values

    return pd.DataFrame(out)


def xcorr_best_lag(lateral_gr: np.ndarray, tw_gr: np.ndarray,
                   center: int, half_win: int = DTW_WINDOW,
                   lags: list = CORR_LAGS) -> dict:
    """
    At depth index `center`, extract a local GR window from the lateral
    and find the typewell lag that maximises Pearson correlation.
    Returns dict: best_lag, best_corr, mean_corr
    """
    lo  = max(0, center - half_win)
    hi  = min(len(lateral_gr), center + half_win)
    win = lateral_gr[lo:hi]
    if len(win) < 5:
        return {'xcorr_best_lag': 0.0, 'xcorr_best_corr': 0.0, 'xcorr_mean_corr': 0.0}

    corrs = []
    for lag in lags:
        tlo = max(0, lo + lag)
        thi = min(len(tw_gr), hi + lag)
        if thi - tlo != len(win):
            corrs.append(0.0)
            continue
        tw_win = tw_gr[tlo:thi]
        if tw_win.std() < 1e-6 or win.std() < 1e-6:
            corrs.append(0.0)
        else:
            corrs.append(np.corrcoef(win, tw_win)[0, 1])

    best_idx = int(np.argmax(corrs))
    return {
        'xcorr_best_lag':  float(lags[best_idx]),
        'xcorr_best_corr': float(corrs[best_idx]),
        'xcorr_mean_corr': float(np.mean(corrs)),
    }


def dtw_features(lateral_gr: np.ndarray, tw_gr: np.ndarray,
                 center: int, half_win: int = DTW_WINDOW) -> dict:
    """
    Compute DTW distance between local lateral GR window and the
    corresponding typewell window at the best-corr lag offset.
    """
    lo  = max(0, center - half_win)
    hi  = min(len(lateral_gr), center + half_win)
    win_lat = lateral_gr[lo:hi].astype(np.float64)
    win_tw  = tw_gr[lo:hi].astype(np.float64)

    if len(win_lat) < 5 or np.isnan(win_tw).any():
        return {'dtw_dist': np.nan, 'dtw_norm_dist': np.nan}

    # Normalise windows before DTW so amplitude differences don't dominate
    def norm01(x):
        r = x.max() - x.min()
        return (x - x.min()) / (r + 1e-9)

    d = dtw.distance_fast(norm01(win_lat), norm01(win_tw), window=half_win // 2)
    return {
        'dtw_dist':      float(d),
        'dtw_norm_dist': float(d / len(win_lat)),
    }


def build_well_features(well_df: pd.DataFrame,
                        typewell_df,
                        md_col=COL_MD, gr_col=COL_GR,
                        tvt_col=COL_TVT) -> pd.DataFrame:
    """Full feature matrix for one well."""
    well = well_df.sort_values(md_col).copy()
    md   = well[md_col].values
    gr   = well[gr_col].fillna(method='ffill').fillna(method='bfill').values

    # Typewell GR interpolated to this well's MD grid
    tw_gr = interpolate_typewell_gr(typewell_df, md)

    # A + B: depth + rolling GR
    feat = rolling_gr_features(gr, md)

    # C: correlation and DTW features (computed per row — vectorised over depth)
    xcorr_rows = [xcorr_best_lag(gr, tw_gr, i) for i in range(len(md))]
    dtw_rows   = [dtw_features(gr, tw_gr, i)   for i in range(len(md))]

    feat = pd.concat([
        feat,
        pd.DataFrame(xcorr_rows),
        pd.DataFrame(dtw_rows),
    ], axis=1)

    # Typewell GR value at each depth (as direct feature)
    feat['tw_gr'] = tw_gr

    # Difference between lateral GR and typewell GR
    feat['gr_tw_diff']  = gr - tw_gr
    feat['gr_tw_ratio'] = gr / (tw_gr + 1e-9)

    # Target
    if tvt_col in well.columns:
        feat['tvt'] = well[tvt_col].values

    feat['well_id'] = well[COL_WELL].values

    return feat


print("Feature engineering functions defined.")
print("Testing on one well...")
sample_well_df  = train_df[train_df[COL_WELL] == sample_well]
sample_feat     = build_well_features(sample_well_df, typewell_df)
print(f"Feature shape for well '{sample_well}': {sample_feat.shape}")
print(sample_feat.head(2))

In [ ]:
# ── Build full feature matrix (all wells) ─────────────────────────────────
print("Building features for all wells (this takes a few minutes)...")
all_feats = []
for wid in train_df[COL_WELL].unique():
    wdf = train_df[train_df[COL_WELL] == wid]
    f   = build_well_features(wdf, typewell_df)
    all_feats.append(f)
    print(f"  {wid}: {len(f)} rows")

feat_df = pd.concat(all_feats, ignore_index=True)
print(f"\nFull feature matrix: {feat_df.shape}")
print(f"Missing values:\n{feat_df.isnull().sum()[feat_df.isnull().sum()>0]}")

## 4. Group K-Fold cross-validation

**Critical:** split by `well_id`, never by row. Adjacent depth samples from the same well
are highly correlated — a random split gives fraudulently good CV scores.

In [ ]:
DROP_COLS = ['tvt', 'well_id']
FEATURE_COLS = [c for c in feat_df.columns if c not in DROP_COLS]

X   = feat_df[FEATURE_COLS].copy()
y   = feat_df['tvt'].copy()
grp = feat_df['well_id'].copy()

# Fill any residual NaNs with column median
X = X.fillna(X.median())

print(f"X shape: {X.shape}  |  Features: {len(FEATURE_COLS)}")
print(f"Target range: {y.min():.2f} → {y.max():.2f}")
print(f"Groups: {grp.nunique()} wells")

In [ ]:
# ── LightGBM parameters ────────────────────────────────────────────────────
# monotone_constraints: +1 means TVT should increase with measured depth
# (geologically, we expect smooth monotone increase along the lateral).
# Find the index of the 'md' feature for the constraint.
md_feat_idx = FEATURE_COLS.index('md') if 'md' in FEATURE_COLS else -1
mono = [0] * len(FEATURE_COLS)
if md_feat_idx >= 0:
    mono[md_feat_idx] = 1   # TVT weakly increases with depth

LGB_PARAMS = dict(
    objective         = 'regression_l1',   # MAE objective — robust to outlier interpretations
    metric            = 'rmse',
    n_estimators      = 2000,
    learning_rate     = 0.03,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    monotone_constraints = mono,
    n_jobs            = -1,
    random_state      = SEED,
    verbose           = -1,
)

print("LightGBM params set.")
print(f"Monotone constraint on 'md' feature: {mono[md_feat_idx] if md_feat_idx >= 0 else 'N/A'}")

In [ ]:
gkf     = GroupKFold(n_splits=N_FOLDS)
oof     = np.zeros(len(y))
models  = []   # one model per fold
fold_rmse = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, grp)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    val_wells = grp.iloc[va_idx].unique()
    print(f"\nFold {fold+1}/{N_FOLDS}  |  val wells: {list(val_wells)}")

    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set  = [(X_va, y_va)],
        callbacks = [
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )

    oof[va_idx] = model.predict(X_va)
    rmse = mean_squared_error(y_va, oof[va_idx], squared=False)
    fold_rmse.append(rmse)
    print(f"  Fold {fold+1} RMSE: {rmse:.4f}  |  best iter: {model.best_iteration_}")
    models.append(model)

oof_rmse = mean_squared_error(y, oof, squared=False)
print(f"\n{'='*50}")
print(f"OOF RMSE (all folds): {oof_rmse:.4f}")
print(f"Per-fold:             {[f'{r:.4f}' for r in fold_rmse]}")

## 5. Savitzky-Golay post-processing

TVT must be physically smooth — abrupt jumps are geologically implausible.
SG filter preserves the shape of the curve while removing high-frequency noise.

In [ ]:
def smooth_tvt(preds: np.ndarray, window: int = 31, poly: int = 3) -> np.ndarray:
    """
    Apply Savitzky-Golay smoothing per well.
    window must be odd and > poly.
    """
    if len(preds) < window:
        return preds
    return savgol_filter(preds, window_length=window, polyorder=poly)


# Apply per-well smoothing to OOF predictions
oof_smooth = oof.copy()
for wid in grp.unique():
    mask = (grp == wid).values
    oof_smooth[mask] = smooth_tvt(oof[mask])

oof_smooth_rmse = mean_squared_error(y, oof_smooth, squared=False)
print(f"OOF RMSE before smoothing : {oof_rmse:.4f}")
print(f"OOF RMSE after  smoothing : {oof_smooth_rmse:.4f}")

## 6. Final model — retrain on all data

We retrain a single model on 100% of the training data using the median
best_iteration across folds. This is the model we freeze and export.

In [ ]:
best_iters = [m.best_iteration_ for m in models]
final_n_est = int(np.median(best_iters))
print(f"Best iterations per fold: {best_iters}")
print(f"Final model n_estimators:  {final_n_est}")

final_params = {**LGB_PARAMS, 'n_estimators': final_n_est}
final_params.pop('verbose', None)

final_model = lgb.LGBMRegressor(**final_params)
final_model.fit(X, y)
print("Final model trained on all data.")

# Feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)
print("\nTop 20 features:")
print(importances.head(20))

## 7. Export — pickle pipeline + ONNX

We wrap the model in a lightweight `InferenceModel` dataclass that carries:
- the LightGBM model
- the expected feature column names (order matters for ONNX)
- the post-processing smoothing parameters
- the typewell GR array (needed at inference time)

This makes the pickle fully self-contained for the Drillbotics runtime.

In [ ]:
import dataclasses
from typing import Optional

@dataclasses.dataclass
class TVTInferenceModel:
    """
    Self-contained TVT estimator.
    Call .predict(well_df, typewell_df) from the Drillbotics controller.
    """
    model:          lgb.LGBMRegressor
    feature_cols:   list
    sg_window:      int   = 31
    sg_poly:        int   = 3
    oof_rmse:       float = 0.0
    n_train_wells:  int   = 0

    def predict(
        self,
        well_df: pd.DataFrame,
        typewell_df,
        smooth: bool = True,
    ) -> np.ndarray:
        """
        Parameters
        ----------
        well_df     : DataFrame with columns COL_MD and COL_GR at minimum
        typewell_df : Reference typewell DataFrame (same format as training)
        smooth      : Apply Savitzky-Golay smoothing to output

        Returns
        -------
        tvt_pred : np.ndarray, same length as well_df
        """
        feat = build_well_features(well_df, typewell_df)
        X_inf = feat[self.feature_cols].fillna(feat[self.feature_cols].median())
        preds = self.model.predict(X_inf)
        if smooth:
            preds = smooth_tvt(preds, self.sg_window, self.sg_poly)
        return preds


inference_model = TVTInferenceModel(
    model         = final_model,
    feature_cols  = FEATURE_COLS,
    sg_window     = 31,
    sg_poly       = 3,
    oof_rmse      = oof_smooth_rmse,
    n_train_wells = train_df[COL_WELL].nunique(),
)

# ── Pickle export ──────────────────────────────────────────────────────────
pickle_path = OUTPUT_DIR / 'tvt_model.pkl'
with open(pickle_path, 'wb') as f:
    pickle.dump(inference_model, f, protocol=5)
print(f"Pickle saved → {pickle_path}  ({pickle_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
# ── ONNX export ────────────────────────────────────────────────────────────
# LightGBM → ONNX via skl2onnx (or lgb's native onnx export if available)
# We use skl2onnx for compatibility with the onnxruntime edge runtime.

onnx_path = OUTPUT_DIR / 'tvt_model.onnx'
n_features = len(FEATURE_COLS)

try:
    # skl2onnx path (works with lgb via the lightgbm onnx converter)
    from skl2onnx.common.shape_calculator import calculate_linear_regressor_output_shapes
    from onnxmltools.convert.lightgbm.operator_converters.LightGbm import convert_lightgbm

    update_registered_converter(
        lgb.LGBMRegressor,
        'LightGbmLGBMRegressor',
        calculate_linear_regressor_output_shapes,
        convert_lightgbm,
        options={'nocl': [True, False], 'zipmap': [True, False, 'columns']}
    )

    initial_type = [('float_input', FloatTensorType([None, n_features]))]
    onnx_model   = convert_sklearn(final_model, initial_types=initial_type, target_opset=17)

    with open(onnx_path, 'wb') as f:
        f.write(onnx_model.SerializeToString())
    print(f"ONNX saved  → {onnx_path}  ({onnx_path.stat().st_size / 1024:.1f} KB)")

except Exception as e:
    # Fallback: use LightGBM's native ONNX export (lgb >= 4.0)
    print(f"skl2onnx path failed ({e}), trying lgb native export...")
    try:
        final_model.booster_.dump_model()   # verify booster exists
        # lgb native: save as text then convert via onnxmltools CLI
        lgb_model_path = OUTPUT_DIR / 'tvt_model.txt'
        final_model.booster_.save_model(str(lgb_model_path))
        print(f"LGB text model saved → {lgb_model_path}")
        print("To convert: run  `python -m onnxmltools.convert.main --input tvt_model.txt`")
        print("OR use the Hugging Face Optimum CLI: `optimum-cli export onnx ...`")
        onnx_path = lgb_model_path   # update reference
    except Exception as e2:
        print(f"Both ONNX paths failed: {e2}")
        print("The pickle model is fully functional; ONNX can be exported in a separate step.")

## 8. Sanity checks

### 8a. Inference latency — must be <15 min total for Drillbotics judging
### 8b. Round-trip pickle load + predict
### 8c. ONNX runtime check

In [ ]:
# ── 8a. Latency ────────────────────────────────────────────────────────────
sample_wdf  = train_df[train_df[COL_WELL] == sample_well]
sample_feat = build_well_features(sample_wdf, typewell_df)
X_sample    = sample_feat[FEATURE_COLS].fillna(sample_feat[FEATURE_COLS].median())

N_REPEATS = 50
t0 = time.perf_counter()
for _ in range(N_REPEATS):
    _ = final_model.predict(X_sample)
elapsed_ms = (time.perf_counter() - t0) / N_REPEATS * 1000
print(f"LGB inference latency: {elapsed_ms:.2f} ms  ({len(X_sample)} depth samples)")

per_sample_us = elapsed_ms / len(X_sample) * 1000
print(f"Per-sample: {per_sample_us:.2f} µs  → well within real-time D-WIS loop budget")

In [ ]:
# ── 8b. Pickle round-trip ──────────────────────────────────────────────────
with open(pickle_path, 'rb') as f:
    loaded_model = pickle.load(f)

preds_loaded = loaded_model.predict(sample_wdf, typewell_df, smooth=True)
print(f"Pickle round-trip: OK — {len(preds_loaded)} predictions")
print(f"TVT pred range: {preds_loaded.min():.2f} → {preds_loaded.max():.2f}")
print(f"Model metadata: OOF RMSE={loaded_model.oof_rmse:.4f}, "
      f"trained on {loaded_model.n_train_wells} wells")

In [ ]:
# ── 8c. ONNX runtime check ─────────────────────────────────────────────────
if onnx_path.suffix == '.onnx' and onnx_path.exists():
    sess = ort.InferenceSession(str(onnx_path))
    input_name  = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name

    X_onnx = X_sample.values.astype(np.float32)
    onnx_preds = sess.run([output_name], {input_name: X_onnx})[0].flatten()

    max_diff = np.abs(final_model.predict(X_sample) - onnx_preds).max()
    print(f"ONNX round-trip: OK — max diff vs LGB: {max_diff:.6f}")

    # ONNX latency
    t0 = time.perf_counter()
    for _ in range(N_REPEATS):
        _ = sess.run([output_name], {input_name: X_onnx})
    onnx_ms = (time.perf_counter() - t0) / N_REPEATS * 1000
    print(f"ONNX inference latency: {onnx_ms:.2f} ms")
else:
    print("ONNX file not available (see export step above for next steps).")

## 9. Competition submission (Kaggle)

Generate the submission CSV for the leaderboard using all fold models
with a simple average ensemble.

In [ ]:
test_path = DATA_DIR / 'test.csv'

if test_path.exists():
    test_df = normalise_cols(pd.read_csv(test_path))
    print(f"Test shape: {test_df.shape}")

    test_wells = []
    for wid in test_df[COL_WELL].unique():
        wdf  = test_df[test_df[COL_WELL] == wid]
        feat = build_well_features(wdf, typewell_df)
        test_wells.append(feat)

    test_feat = pd.concat(test_wells, ignore_index=True)
    X_test    = test_feat[FEATURE_COLS].fillna(test_feat[FEATURE_COLS].median())

    # Ensemble: mean of all fold models
    test_preds_folds = np.column_stack([m.predict(X_test) for m in models])
    test_preds = test_preds_folds.mean(axis=1)

    # Per-well smoothing
    for wid in test_feat['well_id'].unique():
        mask = (test_feat['well_id'] == wid).values
        test_preds[mask] = smooth_tvt(test_preds[mask])

    sub = test_df[[COL_WELL, COL_MD]].copy()
    sub['tvt'] = test_preds

    sub_path = OUTPUT_DIR / 'submission.csv'
    sub.to_csv(sub_path, index=False)
    print(f"Submission saved → {sub_path}")
    print(sub.head())
else:
    print("No test.csv found — skipping submission generation.")

## 10. Drillbotics usage guide

```python
# ── In your Drillbotics Case 2 controller ────────────────────────────────
import pickle, numpy as np, pandas as pd

# Load once at startup
with open('tvt_model.pkl', 'rb') as f:
    tvt_estimator = pickle.load(f)

# At each D-WIS polling cycle (e.g. 1 Hz):
def estimate_tvt(live_gr_buffer: list, live_md_buffer: list, typewell_df) -> float:
    """
    live_gr_buffer : last N GR readings from D-WIS (e.g. last 120 samples)
    live_md_buffer : corresponding measured depths
    Returns: estimated TVT at the current bit position
    """
    well_df = pd.DataFrame({'md': live_md_buffer, 'gr': live_gr_buffer})
    well_df['well_id'] = 'live'
    preds = tvt_estimator.predict(well_df, typewell_df, smooth=True)
    return float(preds[-1])   # TVT at current bit depth

# ONNX path (lower latency, no Python LGB dependency on edge):
import onnxruntime as ort
sess = ort.InferenceSession('tvt_model.onnx')
# ... (build feature vector same as training, run sess.run(...))
```

### Hardware budget check
| Component | Requirement | This model |
|---|---|---|
| RAM | ≤4–8 GB | ~50 MB |
| CPU | ≤2 vCPU | <1 vCPU |
| Inference latency | <15 min total | <5 ms per well |
| Offline | Required | ✓ (no network calls) |